In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install transformers datasets scikit-learn pandas numpy matplotlib seaborn gradio -q

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertModel, Trainer, TrainingArguments
import warnings
warnings.filterwarnings('ignore')

In [5]:
train_df = pd.read_csv('/content/drive/MyDrive/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/val.csv')
test_df = pd.read_csv('/content/drive/MyDrive/test.csv')

print(f"训练集: {len(train_df)} 条")
print(f"验证集: {len(val_df)} 条")
print(f"测试集: {len(test_df)} 条")
print(f"\n标签分布:\n{train_df['label'].value_counts()}")

训练集: 80000 条
验证集: 5000 条
测试集: 5000 条

标签分布:
label
0    40056
1    39944
Name: count, dtype: int64


In [6]:
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

In [7]:
class MovieDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.values
        self.labels = labels.values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [22]:
from transformers.modeling_outputs import SequenceClassifierOutput

class BertWithExtraLayer(nn.Module):
    def __init__(self, num_classes=2, extra_dim=256):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-chinese')
        self.dropout = nn.Dropout(0.2)
        self.extra_fc = nn.Linear(768, extra_dim)
        self.classifier = nn.Linear(extra_dim, num_classes)
        self.relu = nn.ReLU()
        self.num_labels = num_classes

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.pooler_output

        x = self.dropout(cls_output)
        x = self.extra_fc(x)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None
        )

In [23]:
train_dataset = MovieDataset(train_df['text'], train_df['label'])
val_dataset = MovieDataset(val_df['text'], val_df['label'])
test_dataset = MovieDataset(test_df['text'], test_df['label'])

model = BertWithExtraLayer(num_classes=2, extra_dim=256)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"使用设备: {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


使用设备: cuda


In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [25]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none'
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.334001,0.378961,0.866200,0.868578,0.865476,0.867024
2,0.236510,0.399383,0.872800,0.883550,0.861111,0.872186


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.334001,0.378961,0.866200,0.868578,0.865476,0.867024
2,0.236510,0.399383,0.872800,0.883550,0.861111,0.872186
3,0.095498,0.564998,0.873200,0.875997,0.871825,0.873906


TrainOutput(global_step=15000, training_loss=0.2522542256673177, metrics={'train_runtime': 5057.3015, 'train_samples_per_second': 47.456, 'train_steps_per_second': 2.966, 'total_flos': 0.0, 'train_loss': 0.2522542256673177, 'epoch': 3.0})

In [27]:
results = trainer.evaluate(test_dataset)

print("=" * 50)
print("测试集结果（BERT + 额外全连接层）")
print("=" * 50)
for key, value in results.items():
    if 'eval' in key:
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.095498,0.561809,3,0.875000,0.866200,0.887959,0.876944


测试集结果（BERT + 额外全连接层）
eval_loss: 0.5618
eval_accuracy: 0.8750
eval_precision: 0.8662
eval_recall: 0.8880
eval_f1: 0.8769


In [1]:
# 设置中文字体
import matplotlib.pyplot as plt
import matplotlib

# 下载并使用 SimHei 字体
!wget -q https://github.com/StellarCN/scp_zh/raw/master/fonts/SimHei.ttf
matplotlib.font_manager.fontManager.addfont('SimHei.ttf')
matplotlib.rcParams['font.family'] = 'SimHei'
matplotlib.rcParams['axes.unicode_minus'] = False

print("中文字体设置成功")

中文字体设置成功


In [2]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['负面', '正面'],
            yticklabels=['负面', '正面'])
plt.xlabel('预测')
plt.ylabel('实际')
plt.title('混淆矩阵')
plt.show()

print(f"负面猜对: {cm[0][0]}, 负面猜错: {cm[0][1]}")
print(f"正面猜对: {cm[1][1]}, 正面猜错: {cm[1][0]}")

NameError: name 'trainer' is not defined

In [29]:
save_path = '/content/drive/MyDrive/bert_extra_layer_model'
model.bert.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
torch.save(model.extra_fc.state_dict(), f'{save_path}/extra_fc.pt')
torch.save(model.classifier.state_dict(), f'{save_path}/classifier.pt')
print(f"模型已保存到: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

模型已保存到: /content/drive/MyDrive/bert_extra_layer_model


In [30]:
def predict(text):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1).item()

    return "正面" if pred == 1 else "负面"

import gradio as gr

iface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="请输入电影评论", lines=3, placeholder="例如：这部电影太好看了..."),
    outputs=gr.Label(label="情感分析结果"),
    title="电影评论情感分析系统（BERT + 额外层）",
    description="输入中文电影评论，系统会自动判断是正面还是负面情感。",
    examples=[
        ["这部电影太棒了，演员演技在线！"],
        ["剧情拖沓，特效五毛，失望。"],
        ["还不错，值得一看。"]
    ]
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://561f525057dbfeb8a3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
